In [ ]:
%matplotlib notebook

In [ ]:
%matplotlib qt
#%matplotlib inline

# Confirmation Bias Project
## Behavioural analyses
Launching analysis for each of the experiments: ['3reps', 'CJ']


We recommend to run this script in Jupyter Notebook. It is possible to not visualize some plots in JupyterLab

__Metadprime is computed using the library from:__

pip install git+https://github.com/LegrandNico/metadPy.git

##### Import important functions and libraries

In [ ]:
import os, glob, platform, sys
import numpy as np
import pickle
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import scipy
import scipy.stats as stats
from scipy import signal
import seaborn as sns
from statsmodels.formula.api import ols
import statsmodels.formula.api as smf
from statsmodels.stats.anova import AnovaRM
import pingouin as pg

from statsmodels.stats.multicomp import (pairwise_tukeyhsd, MultiComparison)
from matplotlib.lines import Line2D
import statsmodels as sms
#import ptitprince as pt
pd.options.display.max_columns = None # display all the columns in pandas dataframe
#import plotly.graph_objects as go
#import plotly.express as px
#from plotly.subplots import make_subplots



Metacognitive parameters

In [ ]:
#! pip install numpyro
import pymc as pm
import numpy as np
import arviz as az
#import numpyro
from metadpy.bayesian import hmetad #sometimes is call metadpy not metadPY (it depends on the installed versions)
from metadpy.mle import metad, fit_metad
#from metadPy import load_dataset
from metadpy.sdt import criterion, dprime, rates, roc_auc, scores
import pymer4
# Set the number of cores used by Numpyro here
#numpyro.set_host_device_count(2)
#from metadPy.utils import responseSimulation

#simulation = responseSimulation(d=1, metad=1.5, c=0, 
#                                nRatings=4, nTrials=200)

### My functions

In [ ]:

module_directory = os.path.abspath(os.path.join(os.getcwd(), '../../Helper_funcs')) # Updated path to helper_funcs

if module_directory not in sys.path:
    sys.path.append(module_directory)
    # print(f"Added {module_directory} to sys.path") # Optional: Add a print for confirmation

# --- Standard Import ---
# Get the module object. This imports it the first time or retrieves it if already loaded.
from use_funcs import *


__Plotting functions__

In [ ]:
# function that I used to plot multiple data
def barsplot(data, dx, dy, hue, col, row, pal, size, yaxis, axislabels, sizepoint, dodge):    
    sns.set(font_scale = 1.5, style = 'ticks')         
    ort = "v"; pal = pal; sigma = .5
    g = sns.FacetGrid(data ,  row = row, col = col, height= size['height'], aspect=size['aspect'], margin_titles=True) # col="nrep",    
    if sizepoint == None:
        sizepoint = 6
    if yaxis != None:
        g.set(yaxis['ylim'], yaxis['yticks'])   

    g.map_dataframe(sns.stripplot, x = dx, y = dy, palette = pal, hue=hue, size = sizepoint, edgecolor = "white",
                    linewidth = 0.6, jitter = 0.2, orient = ort,alpha = 0.5, dodge=dodge)
    g.map_dataframe(sns.barplot, x = dx, y = dy, palette = pal, hue=hue,  linewidth = 0.6, orient = ort, dodge=dodge)
    
    #g.map_dataframe(sns.violinplot, x = dx, y = dy,  palette = pal,bw = .5, cut = 0.,scale = "area", width = .6, inner = None, orient = ort, linewidth = 0, zorder = 2)
    
    g.add_legend()

    sns.despine(offset = .5,  trim=True);
    # Set x-axis and y-axis labels
    g.set_axis_labels( axislabels['xlabel'] , axislabels['ylabel'], fontsize = 15 )
    #g.tight_layout()
    return g


__Color scales__

In [ ]:
colpal1 = ["orange","green"]
colpal1a = ["deeppink","deepskyblue"]
#colpal2 = ['black','green','blue']
colpal2 = ['darkorange','lime','royalblue']
colpal1b = ['dodgerblue','navy']
colpal3 = ['green','darkviolet']
colpal3a = ['green','darkviolet','darkgrey']


colpal2 = ['gold','mediumturquoise','darkblue']


In [ ]:
current_dir = os.getcwd()
# Go up 3 levels ("../../../") and then into your target folder
results_path = os.path.abspath(os.path.join(current_dir, "../../../Behav_data/Behavioral/condcisionCJ"))

figures_path = os.path.join(results_path, 'Group_level_analyses','figures') 
# some folders with interesting custom functions

#sys.path.append(path_utils)
#sys.path.append(psychofits)

import psychofit as psy

Concatenating datasets

In [ ]:
df_all = pd.concat([pd.read_csv(os.path.join(results_path,'Behav_data/Behavioral/condcision3reps_same/3reps.csv')),               
pd.read_csv(os.path.join(results_path,'Behav_data/Behavioral/condcisionCJ/CJ.csv'))])
df_all.head()

Creating individual IDs for each participant (to prevent modelling taking different subjects with similar names as identical)

# Recoding new variables

In [ ]:
#Characterizing repeated events in 2 and 3 position
df = df_all.copy()
non_shifted = df[['o1','o2','o3','o4','o5','o6']]
shifted = df[['o1','o2','o3','o4','o5','o6']].shift(axis = 0, periods = 1)
diffmat = (non_shifted == shifted)
diffmat[diffmat == False] = 0
diffmat[diffmat == True] = 1
diffmat = np.mean(diffmat, axis = 1)
diffmat[diffmat < 1 ] = 0


df.insert(6, 'rep', 'diff')   
df.loc[diffmat != 0, 'rep'] = 'rep'
df.loc[df.nrep == 0, 'rep'] = 'diff' # by definition, usually nrep 0 is always different from previous presentations (with exceptions)
# DEPRECATED: df.loc[df.exp_ID == '3diffs', 'rep'] = 'diff' 
df = df.rename(columns={'cond' : 'Stimuli', 'deci': 'Responses',  'confi': 'Confidence', 'correct' : 'accuracy', 'cond-1': 'stimuli1', 
                        'deci-1': 'response1','deci-2': 'response2','deci-3': 'response3',
                        'corr-1' : 'accuracy1','resp' : 'SideResp','RT' : 'rt'})
# Mapping motor actions
df.loc[(df.SideResp == 1) & (df.r_map == 45) |(df.SideResp == -1) & (df.r_map == 0) , 'SideResp'] = 'L' # You should check if this is right or left (See that here is a cursor movement)
df.loc[(df.SideResp == -1) & (df.r_map == 45) |(df.SideResp == 1) & (df.r_map == 0) , 'SideResp'] = 'R'
df.loc[(df.SideResp == 'z') , 'SideResp'] = 'L'
df.loc[(df.SideResp == 'm') , 'SideResp'] = 'R'

df.insert(15, 'SideResp1', np.squeeze(df[['SideResp']]).shift(axis = 0, periods = 1)) 
df.insert(17, 'RepSideResp', np.where(df.SideResp == df.SideResp1, 1, 0))
df.insert(13, 'RepResponses', np.where(df.Responses == df.response1, 1, 0))
df.insert(35, 'Confidence1', df[['Confidence']].shift(axis = 0, periods = 1)) 
df.insert(1, 'id_subj', 0) 

# Adding individual IDs for each participant
exp_ID_l = np.unique(df_all.exp_ID)
id_subj = np.array([])
i = 1  
for ix_expID in exp_ID_l:
    aux = df[(df.exp_ID == ix_expID)]
    subj_l = np.unique(aux.npar)
    for ix_subj in subj_l:
        df.loc[(df.exp_ID == ix_expID) & (df.npar == ix_subj), 'id_subj'] = i
        #nobs = np.shape(aux[(aux.npar == ix_subj)])[0]
        #id_subj = np.append(id_subj, np.ones(nobs) * i)
        #print(i)
        i = i + 1  

#df.insert(1, 'id_subj', id_subj) 
#df.insert(1, 'id_subj', id_subj.astype(int)) 
df['id_subj'] = df['id_subj'].astype('category')
df['nrep'] = df['nrep'].astype('category')
df['ITI'] = df['ITI'].astype('category')

df.loc[df.exp_ID == '3reps', 'trial_type'] = 'repeat'
# DEPRECATED: df.loc[df.exp_ID == '3reps_iEEG', 'trial_type'] = 'repeat'
# DEPRECATED: df.loc[df.exp_ID == '3reps_lag', 'trial_type'] = 'repeat'
# DEPRECATED: df.loc[df.exp_ID == '3diffs', 'trial_type'] = 'nonrepeat'

In [ ]:
df_all[df_all.exp_ID == 'CJ']

Derivating confidence from RT as in Braun et al. 2018

In [ ]:
def calculate_rtconf(group: pd.DataFrame) -> pd.Series:
    rt_series = group['rt']
    log_rt = np.log(rt_series)
    max_log_rt = log_rt.max()
    transformed_rt = max_log_rt - log_rt
    rtconf = (transformed_rt - transformed_rt.mean()) / transformed_rt.std()
    group['rtconf'] = rtconf
    return group


df = df.groupby(['id_subj','exp_ID']).apply(calculate_rtconf).reset_index(drop=True)
df['rtconf1'] = df['rtconf'].shift(1, fill_value  = 0) # deci in trial n-1


__Saving the concatenated dataset for future analysis__

In [ ]:
df.to_csv('./all_data_CSV/cond_df.csv')


# Preparing the data in long format

In [ ]:
# df to long format
dflong = pd.melt(df, id_vars=['exp_ID','id_subj','nrep','rep','nblock','ntrial','stimuli1','Stimuli', 'response2','response1','Responses', 'accuracy1', 'Confidence1','Confidence','rtconf','rtconf1','ITI','trial_type','rt','metad'], value_vars=['d1', 'd2', 'd3', 'd4', 'd5', 'd6'])
dflong.shape
dflong.rename(columns={'variable' : "stim", 'value' : 'dv'}, inplace=True)

out = pd.melt(df, id_vars=['exp_ID','id_subj','nrep','rep','nblock','ntrial','stimuli1','Stimuli', 'response2','response1','Responses', 'accuracy1', 'Confidence1','Confidence','rtconf','rtconf1','ITI','trial_type','rt','metad'], value_vars=['o1', 'o2', 'o3', 'o4', 'o5', 'o6'])
dflong['orient'] = out.value
dflong.orient = np.rad2deg(dflong.orient) # transform to degrees
del out

dflong = dflong.sort_values(['nblock','id_subj','ntrial','nrep',],ascending=[True,True,True,True]) # reorder variables
dflong.reset_index(drop=True, inplace=True)
dflong.insert(5, "stim_cond", "D")
dflong.loc[(dflong.dv < 0), "stim_cond"]  = "C"


dflong.insert(7, 'cong_deci', "I") # if stim is congruent with previous repetition decosion category
# an stimuli is expected when the previous dv and previous deci were the same (i.e.DV==D & pre_deci == D)
dflong.loc[(dflong['response1'] == 0) & (dflong.stim_cond == 'C') | (dflong['response1'] == 1) & (dflong.stim_cond == 'D'), "cong_deci"]  = "C"



dflong.insert(8, 'cong_stim', 'I') # if stim is congruent with previous repetition mean category
dflong.loc[(dflong['response1'] == 0) & (dflong.stim_cond == 'C') | (dflong['response1'] == 'D') & (dflong.stim_cond == 'D'), 'cong_stim'] = 'C'

# relabeling variables
#dflong["cond-1"].replace({0: "C", 1: "D"}, inplace=True)
#dflong["deci"].replace({0: "C", 1: "D"}, inplace=True)
#dflong["deci-1"].replace({0: "C", 1: "D"}, inplace=True)
#dflong["deci-2"].replace({0: "C", 1: "D"}, inplace=True)


dflong['ov'] = signal.sawtooth(2*np.deg2rad(dflong.orient), 0.5) # orientation variable (here I define an ov to make stimuli that differ by less of 45 degrees or more than 135 as similar)
# plt.scatter(dflong.orient,dflong['ov']) # see here


dflong.head(5)

Saving data to fit mixed GLM in R

In [ ]:
dflong.to_csv('./all_data_CSV/cond_dlong.csv')

__Accuracy, dprime and criterion calculation (for all the experiments)__

In [ ]:
group_avg_correct = df.groupby(['exp_ID', 'id_subj','nrep'],as_index=False)[['accuracy','rt']].mean()
group_sdt_avg = df.groupby(['exp_ID', 'id_subj','nrep']).apply(sdt)
group_sdt_avg.reset_index(inplace=True)
group_repeat_correct = df.groupby(['exp_ID', 'id_subj','nrep', 'accuracy'],as_index=False)[['RepResponses']].mean()

# ALL THE EXPERIMENTS TOGETHER

In [ ]:
group_sdt = df.groupby(['exp_ID', 'id_subj','nrep','rep']).apply(sdt)
group_sdt.reset_index(inplace=True)

In [ ]:

sizeplot = {}; sizeplot['height'] = 4; sizeplot['aspect'] = 0.8
axislabels = {}; axislabels['xlabel'] = ' '; axislabels['ylabel']= 'dprime'
yaxis = None # yaxis = {}; yaxis['ylim']=[-2,4], yaxis['yticks']=[-2, 0, 2, 4]
dx = 'nrep'; dy = 'dprime'; hue = 'nrep';  row = None; col = None
pal = colpal2

barsplot(group_sdt_avg, dx, dy, hue, col, row, pal, sizeplot, yaxis, axislabels, sizepoint = 4, dodge = False)

#myviolinplot(df_betas[(df_betas.parameter != 'intercept') ], dx, dy, col, row, pal, sizeplot, yaxis, axislabels, sizepoint = 5)

In [ ]:
sizeplot = {}; sizeplot['height'] = 4; sizeplot['aspect'] = 0.8
axislabels = {}; axislabels['xlabel'] = ' '; axislabels['ylabel']= 'c'
yaxis = None # yaxis = {}; yaxis['ylim']=[-2,4], yaxis['yticks']=[-2, 0, 2, 4]
dx = 'nrep'; dy = 'c'; hue = 'nrep';  row = None; col = None
pal = colpal2

barsplot(group_sdt_avg, dx, dy, hue, col, row, pal, sizeplot, yaxis, axislabels, sizepoint = 4, dodge = False)
#myviolinplot(df_betas[(df_betas.parameter != 'intercept') ], dx, dy, col, row, pal, sizeplot, yaxis, axislabels, sizepoint = 5)

In [ ]:
sizeplot = {}; sizeplot['height'] = 4.5; sizeplot['aspect'] = 0.7
axislabels = {}; axislabels['xlabel'] = ' '; axislabels['ylabel']= 'dprime'
yaxis = None # yaxis = {}; yaxis['ylim']=[-2,4], yaxis['yticks']=[-2, 0, 2, 4]
dx = 'nrep'; dy = 'dprime'; hue = 'rep';  row = None; col = 'exp_ID'
pal = colpal1b

barsplot(group_sdt, dx, dy, hue, col, row, pal, sizeplot, yaxis, axislabels, sizepoint = 7, dodge = True)
#myviolinplot(df_betas[(df_betas.parameter != 'intercept') ], dx, dy, col, row, pal, sizeplot, yaxis, axislabels, sizepoint = 5)

In [ ]:
def cond_bias_func(self):
    qs = pd.qcut(self.rt, q = [0, 0.15, 0.3, 0.5, 0.7, 0.9, 1], labels = False)
    self['qs'] = qs
    condbias = self.groupby(['qs'],as_index=False)[['RepResponses']].mean()
    return condbias

In [ ]:
condbias = df.groupby(['id_subj','nrep','rep','exp_ID']).apply(cond_bias_func)
condbias.reset_index(inplace=True)

In [ ]:
sns.relplot(
   data=condbias, x="qs", y="RepResponses", hue="nrep", col = 'exp_ID',
    kind="line", palette=colpal2
)

In [ ]:
sns.relplot(
   data=condbias, x="qs", y="RepResponses", hue="nrep",
    kind="line", palette=colpal2, height = 4
)

Condition response function

In [ ]:
def compute_condbias(x):
    rt_series = x['rt']

    # Remove or replace zero/negative RTs
    #rt_series = rt_series.clip(lower=1e-5)

    #log_rt = np.log(rt_series)
    #max_log_rt = log_rt.max()
    #transformed_rt = max_log_rt - log_rt

    transformed_rt = rt_series.copy()
    # Handle case where std is zero
    if transformed_rt.std() == 0:
        zRT = pd.Series(0, index=x.index)
    else:
        zRT = (transformed_rt - transformed_rt.mean()) / transformed_rt.std()

    x = x.copy()  # Avoid SettingWithCopyWarning
    x['zRT'] = zRT

    # Bin using qcut with error handling for duplicate edges
    try:
        x['qs'] = pd.qcut(x['zRT'], q=[0, 0.15, 0.3, 0.5, 0.7, 0.9, 1], labels=False, duplicates='drop')
    except ValueError:
        x['qs'] = np.nan  # or set a default value / bin

    return x

# Apply to groups

In [ ]:
dat

In [ ]:
#dat = simdata_lapse_bias.copy()
dat = df.copy()
dat = dat.groupby(['id_subj', 'nrep','response1','exp_ID'], group_keys=False).apply(compute_condbias).reset_index(drop=True)
#outmean = dat.groupby(['id_subj','nrep','response1','qs'],as_index=False)['Responses'].mean()
outmean = dat.groupby(['id_subj','nrep','response1','qs','exp_ID'],as_index=False)['Responses'].mean()
outmean.loc[outmean['nrep'] == 0, 'response1'] = 3.0 # Use a float if original is float


In [ ]:
outmean

In [ ]:
sns.set(font_scale = 1, style = 'ticks')     
# Create the plot
dat = outmean[outmean['exp_ID'] == '3reps']
g = sns.relplot(
   data=dat, x="qs", y="Responses", hue="response1",
   kind="line", palette=colpal3a, aspect=1, height=2.5
)

# --- Modifications for x-axis ticks and label ---
# Set specific x-axis tick marks
g.set(xticks=[0, 2.5, 5])
g.set(yticks=[0, 0.5, 1])

# Set the x-axis label with a small font size
# Iterate over the axes in the FacetGrid (though for a single plot, there's one)
for ax in g.axes.flat:
    ax.set_xlabel("q(RTs)", fontsize=10) # 'small', 'x-small', or a specific number like 8 or 10
    ax.set_ylabel("P(Diagonal)", fontsize=10) # 'small', 'x-small', or a specific number like 8 or 10

if g._legend:
    # Set the legend title
    g._legend.set_title("N-1 Choice")
    
    legend_handles = g._legend.legendHandles if hasattr(g._legend, 'legendHandles') else g._legend.legend_handles # older vs newer matplotlib
    current_labels_from_handles = [handle.get_label() for handle in legend_handles]
    
    # Option 1: If you know the exact order of hue categories in the legend
    # (Often sorted numerically/alphabetically by default)
    # Example: if pre_deci values 0.0, 1.0, 3.0 map to legend entries in that order:
    new_labels = ['Cardinal', 'Diagonal', 'Unbiased'] # Check this order!

    # Option 2: More robust mapping if pre_deci values are known
    # This requires knowing what your `pre_deci` values are.
    # Let's say your unique `pre_deci` values used for hue are [0.0, 1.0, 3.0]
    # and they appear in the legend in this sorted order.
    
    label_mapping = {
        # Convert pre_deci values to string as they appear in legend texts if they are numeric
        str(float(0.0)): 'Cardinal', # Assuming pre_deci was 0.0
        str(float(1.0)): 'Diagonal', # Assuming pre_deci was 1.0
        str(float(3.0)): 'Unbiased'  # The value you set for nrep==0
    }

    if len(g._legend.texts) == len(new_labels): # Basic check
         for i, text_obj in enumerate(g._legend.texts):
            original_label_text = text_obj.get_text()
            # Try to map, fallback to predefined order if direct map fails
            if original_label_text in label_mapping:
                text_obj.set_text(label_mapping[original_label_text])
            elif i < len(new_labels): # Fallback to ordered list if key not found
                 print(f"Warning: Label '{original_label_text}' not in mapping, using ordered label '{new_labels[i]}'. Check label_mapping keys.")
                 text_obj.set_text(new_labels[i])
   
    
    sns.move_legend(
        g, "upper right",

    )
    # Re-apply title and font size as move_legend might recreate it
    if g._legend: # Check if legend still exists after move_legend
        g._legend.set_title("Choice (n-1)")
        plt.setp(g._legend.get_title(), fontsize=10)
        for text_obj in g._legend.texts:
            text_obj.set_fontsize(7)

    

    # Adjust legend properties if needed (e.g., font size of title)
    plt.setp(g._legend.get_title(), fontsize=10) # Adjust title font size


figpath = os.path.join(figures_path, '3reps_cond_resp_function.pdf')
plt.savefig(figpath ,bbox_inches='tight', dpi = 300)

In [ ]:
sns.relplot(
   data=outmean, x="qs", y="RepResponses", hue="nrep",
    kind="line",  palette= colpal2, aspect=1.5, height=1.75
)

LA FIGURA DE CONDITIONAL BIAS SE HACE CALCULANDO LA PROBABILIDAD DE REPETIR EL SESGO DE CADA PARTICIPANTE.
POR ELLO HAY QUE CLASIFICAR PRIMERO A LOS PARTICIPANTS COMO REPEATERS OR ALTERNATORS, Y LUEGO VER SI EN UN TRIAL HAN SEGUIDO SU BIAS
POR EJEMPLO, UN REPEATER QUE REPITE 1 o UN ALTERNATOR QUE ALTERNA 1 TB. EN LAS X QUIZAS HAY QUE BUSCAR LOS PERCENTILES Y TRANSFORMAR LOS RT EN ESCALA LOGARITMICA

# Plotting results for each experiment

## Experiment 3reps

In [ ]:
savefig(fname, *, dpi='figure', format=None, metadata=None,
        bbox_inches=None, pad_inches=0.1,
        facecolor='auto', edgecolor='auto',
        backend=None, **kwargs
       )

In [ ]:
# plotting accurary 


sizepoint = 4
rt = "v"; pal = pal; sigma = .5
dodge = False
pal = colpal2

mycol = colpal2
dat = df[df.exp_ID == '3reps']
sns.set_palette(colpal2)

fig, axes = plt.subplots(1, 4, figsize=(12, 3))

dat = group_avg_correct[group_avg_correct.exp_ID == '3reps']
g = sns.barplot(data=dat, x="nrep", y="accuracy", hue="nrep", ax = axes[0],alpha = 0.9, dodge=dodge,linewidth = 0.6)
g = sns.stripplot(data=dat, x="nrep", y="accuracy", hue="nrep", ax = axes[0],alpha = 0.5, dodge=dodge,edgecolor = "white", linewidth=0.6, jitter = 0.2, legend = False)
g.set_xlabel("nrep",fontsize=10)
g.set_ylabel("P(correct)",fontsize=10)
g.legend(loc='upper left', fontsize=0,  framealpha= 0.0) #labels=['P1', 'P2', 'P3'], #


axes[0].set_ylim(-0.1,1.1)
axes[0].set_yticks([0.0,0.5,1]) 
axes[0].tick_params(axis='y', labelsize=10) 

axes[0].set_xticks([0,1,2])
axes[0].set_xticklabels(['P1','P2','P3'])
axes[0].tick_params(axis='x', labelsize=10) 
axes[0].set_xlabel('Repetition', fontdict={'size':10}); 

sns.despine(offset = .5,  trim=True, ax = axes[0]);


dat = df[df.exp_ID == '3reps']

nrep_labels = np.unique(dat.nrep) #nrep
id_subj_labels = np.unique(dat.id_subj) #npar


axes[1].axvline(0, ls ='--', color= 'black', alpha=0.2)
axes[1].axhline(0.5, ls ='--', color= 'black', alpha=0.2)

for i in id_subj_labels: #for loop to compute the average by each participant
    dat2=dat.loc[(dat.id_subj == i) ,:]
    for cell in nrep_labels:
        sns.set_palette(colpal2)
        sns.regplot(x="rDV", y="Responses",  data=dat2.loc[dat2.nrep == cell,:],
           logistic=True, y_jitter=0, scatter_kws={'alpha':0}, ci=True, n_boot=1, ax = axes[1], label=cell,  truncate=True, 
                         line_kws ={'alpha':0.35, 'lw':0.4}); #mean all subject

for cell in nrep_labels: #for loop to compute the plot by the average sample
    sns.set_palette(colpal2)
    sns.regplot(x="rDV", y="Responses",  data=dat.loc[(dat.nrep == cell) ,:],
           logistic=True, y_jitter=0, scatter_kws={'alpha':0}, ci=True, n_boot=1,   ax = axes[1], label=cell,  truncate=True, 
                     line_kws ={'lw':2.4});
#sns.despine(offset=1, trim=True, ax = axes[j]);


axes[1].set_ylabel('P(diagonal)', fontsize = 10, labelpad=10); #axes[j].set_yticks(np.arange(0, 1.1, step=0.25), fontsize = 15) 
axes[1].set_xlabel('Decision Variable', fontdict={'size':10}); 

#xticks = [-0.6,-0.3,0,0.3,0.6] #np.round(np.arange(-0.9, 0.6, step=0.3),decimals = 2)
#yticks = np.arange(-0., 1.1, step=0.25)

axes[1].set_xlim(-0.72,0.62)
axes[1].set_xticks([-0.6,0,0.6]) 
axes[1].tick_params(axis='x', labelsize=10)  

axes[1].set_yticks([0.0,0.5,1]) 
axes[1].set_ylim(-0.1,1.05)
axes[1].tick_params(axis='y', labelsize=10) 

sns.despine(ax= axes[1], offset=0.5, trim = True);
#'lower right', borderpad=0.1,prop={'size':10}

lines = [Line2D([0], [0], color=c, linewidth=4) for c in colpal2]; labels = ['P1', 'P2', 'P3']; 
axes[1].legend(lines, labels,loc = 'lower right', fontsize=8, framealpha= 0.0)
fig.tight_layout()
#plt.savefig('deci_x_nrep.png',bbox_inches='tight')

# plotting dprime and criterion 



dat = group_sdt_avg[group_sdt_avg.exp_ID == '3reps']
g = sns.barplot(data=dat, x="nrep", y="dprime", hue="nrep", ax = axes[2],alpha = 0.9, dodge=dodge,linewidth = 0.6)
g = sns.stripplot(data=dat, x="nrep", y="dprime", hue="nrep", ax = axes[2],alpha = 0.5, dodge=dodge,edgecolor = "white", linewidth=0.6, jitter = 0.2, legend = False)
g.set_xlabel("nrep",fontsize=10)
g.set_ylabel("dprime",fontsize=10)
g.legend(loc='upper left', fontsize=0,  framealpha= 0.0) #labels=['P1', 'P2', 'P3'], #


axes[2].set_ylim(-0.1,1.85)
axes[2].set_yticks([0.0,0.5,1, 1.5]) 
axes[2].tick_params(axis='y', labelsize=10) 

axes[2].set_xticks([0,1,2])
axes[2].set_xticklabels(['P1','P2','P3'])
axes[2].tick_params(axis='x', labelsize=10) 
axes[2].set_xlabel('Repetition', fontdict={'size':10}); 

sns.despine(offset = .5,  trim=True, ax = axes[2]);

#g.set_axis_labels( axislabels['xlabel'] , axislabels['ylabel'], fontsize = 15 )

dat = group_sdt_avg[group_sdt_avg.exp_ID == '3reps']

g = sns.barplot(data=dat, x="nrep", y="c", hue="nrep", ax = axes[3],alpha = 0.9, dodge=dodge,linewidth = 0.6)
g = sns.stripplot(data=dat, x="nrep", y="c", hue="nrep", ax = axes[3],alpha = 0.5, dodge=dodge,edgecolor = "white", linewidth=0.6,jitter = 0.2, legend = False)
# y axis format
axes[3].set_ylim(-1.1, 0.1)
axes[3].set_yticks([-1,-0.5,0]) 
axes[3].tick_params(axis='y', labelsize=10) 

axes[3].set_xticks([0,1,2])
axes[3].set_xticklabels(['P1','P2','P3'])
axes[3].tick_params(axis='x', labelsize=10) 
axes[3].set_xlabel('Repetition', fontdict={'size':10}); 

# x axis format
#axes[2].tick_params(axis='x', labelsize=10) 
sns.despine(offset = .5,  trim=True, ax = axes[3]);
g.set_xlabel("nrep",fontsize=10)
g.set_ylabel("c",fontsize=10)
g.legend(loc='upper left', fontsize=0,  framealpha= 0.0) #labels=['P1', 'P2', 'P3'], 
#g.set(yaxis['ylim'], yaxis['yticks'])   

figpath = os.path.join(figures_path, '3reps_performance.pdf')
plt.savefig(figpath ,bbox_inches='tight', dpi = 300)


## Previous responses effect

In [ ]:

sns.set(font_scale = 1, style = 'ticks') 
mycol = colpal3
sns.set_palette(mycol)

titles = ['P1', 'P2', 'P3']
pre_deci_labels = np.unique(df['response1'])     #pre_deci
nreps = np.unique(df['nrep'])     #nreps


dat = df[df.exp_ID == '3reps']

nrep_labels = np.unique(dat.nrep) #nrep
id_subj_labels = np.unique(dat.id_subj) #npar


#df_subj = df[df.subj != 's22'] # s16 does not converge
fig, axes = plt.subplots(1, 4, figsize=(12, 3), gridspec_kw={'width_ratios': [.20, .20,.20,.30]})
#fig.suptitle("DV for previous decision for the different repetitions", fontsize=20)

labels = ['(n-1) Cardinal', '(n-1) Diagonal']; 
lines = [Line2D([0], [0], color=c, linewidth=2) for c in mycol]; 
axes[0].legend(lines, labels,loc = 'lower right', fontsize=8, framealpha= 0.0)

for j in nreps:
    axes[j].axvline(0, ls='--', color= 'black', alpha = 0.2)
    axes[j].axhline(0.5, ls='--', color= 'black', alpha = 0.2)
    for cell in pre_deci_labels:
        for i in id_subj_labels: #this for loop makes the plot for each participant 
            dati = dat[(dat.id_subj == i) & (dat['response1'] == cell) & (dat.nrep == j)].copy()#df.loc[(df.npar == i) & (df['deci-1'] == cell) & (df.nrep == j),:]
            sns.regplot(x="rDV", y="Responses",  data= dati,
               logistic=True, y_jitter=0, scatter_kws={'alpha':0}, ci=True, n_boot=1,  
                              label=cell,  truncate=True, line_kws ={'alpha':0.35, 'lw':0.7}, ax = axes[j], color = mycol[cell]);

        datall = dat[(dat['response1'] == cell) & (dat.nrep == j) ].copy()
        sns.regplot(x="rDV", y="Responses",  data= datall,
               logistic=True, y_jitter=.0, scatter_kws={'alpha':0}, ci=True, n_boot=1,  
                          label=cell, truncate=True, line_kws={'lw':2.4}, ax = axes[j], color = mycol[cell]);

    # Tweaking subplots
    axes[j].set_title(titles[j],fontsize = 10)

    axes[j].set_xlabel('DV', fontsize = 10)
    axes[j].set_ylabel('P(Diagonal)', fontsize = 10)

    axes[j].set_xlim(-0.72,0.62)
    axes[j].set_xticks([-0.6,0,0.6]) 

    axes[j].tick_params(axis='x', labelsize=10)   
    axes[j].set_yticks([0.0,0.5,1]) 
    axes[j].set_ylim(-0.1,1.05)
    axes[j].tick_params(axis='y', labelsize=10) 
    #axes[j].xticks(np.arange(-0.6, 0.61, step=0.3), fontsize = 25); 
    sns.despine(ax= axes[j], offset=0.5, trim = True);


fig.tight_layout()
##axes[0].ylabel('p(diagonal)', fontsize = 20, labelpad=20); axes[0].yticks(np.arange(0, 1.1, step=0.25), fontsize = 25); plt.xlabel(' ', fontsize = 0)


sizepoint = 4
rt = "v"; pal = pal; sigma = .5
dodge = False
pal = colpal2

dat = group_repeat_correct[group_repeat_correct.exp_ID == '3reps']
colpal4 = ['crimson', 'mediumaquamarine']
g = sns.barplot(data=dat, x="nrep", y="RepResponses", hue="accuracy", palette = colpal4, ax = axes[3],alpha = 0.9, dodge=True,linewidth = 0.6)
g = sns.stripplot(data=dat, x="nrep", y="RepResponses", hue="accuracy", palette = colpal4, ax = axes[3],alpha = 0.5, dodge=True,edgecolor = "white", linewidth=0.6,jitter = 0.2, legend = False)
# y axis format
axes[3].set_ylim(-0.1, 1.1)
axes[3].set_yticks([0,0.5,1]) 
axes[3].tick_params(axis='y', labelsize=10) 

axes[3].set_xticks([0,1,2])
axes[3].set_xticklabels(['P1','P2','P3'])
axes[3].tick_params(axis='x', labelsize=10) 
axes[3].set_xlabel('Repetition', fontdict={'size':10}); 

# x axis format
#axes[2].tick_params(axis='x', labelsize=10) 
sns.despine(offset = .5,  trim=True, ax = axes[3]);
g.set_xlabel("nrep",fontsize=10)
g.set_ylabel("P(Repeat)",fontsize=10)
#g.legend(loc='upper left', labels = ['correct', 'incorrect'], fontsize=5,  framealpha= 0.0) #labels=['P1', 'P2', 'P3'], 
#g.set(yaxis['ylim'], yaxis['yticks'])   


# modify the legend
handles, labels = g.get_legend_handles_labels()
new_labels = ['Incorrect', 'Correct']
g.legend(handles, new_labels, loc='upper left', title='', fontsize=6, frameon=True, framealpha=0, facecolor='white')

figpath = os.path.join(figures_path, '3reps_prev_resp.pdf')
plt.savefig(figpath ,bbox_inches='tight', dpi = 300)


In [ ]:
dflong.head(10)

## Experiment 3diffs

In [ ]:
# DEPRECATED: EXP = '3diffs'
# plotting accurary 


sizepoint = 4
rt = "v"; pal = pal; sigma = .5
dodge = False
pal = colpal2

mycol = colpal2
dat = df[df.exp_ID == EXP]
sns.set_palette(mycol)

fig, axes = plt.subplots(1, 4, figsize=(12, 3))

dat = group_avg_correct[group_avg_correct.exp_ID == EXP]
g = sns.barplot(data=dat, x="nrep", y="accuracy", hue="nrep", ax = axes[0],alpha = 0.9, dodge=dodge,linewidth = 0.6)
g = sns.stripplot(data=dat, x="nrep", y="accuracy", hue="nrep", ax = axes[0],alpha = 0.5, dodge=dodge,edgecolor = "white", linewidth=0.6, jitter = 0.2, legend = False)
g.set_xlabel("nrep",fontsize=10)
g.set_ylabel("P(correct)",fontsize=10)
g.legend(loc='upper left', fontsize=0,  framealpha= 0.0) #labels=['P1', 'P2', 'P3'], #


axes[0].set_ylim(-0.1,1.1)
axes[0].set_yticks([0.0,0.5,1]) 
axes[0].tick_params(axis='y', labelsize=10) 

axes[0].set_xticks([0,1,2])
axes[0].set_xticklabels(['P1','P2','P3'])
axes[0].tick_params(axis='x', labelsize=10) 
axes[0].set_xlabel('Repetition', fontdict={'size':10}); 

sns.despine(offset = .5,  trim=True, ax = axes[0]);


dat = df[df.exp_ID == EXP]

nrep_labels = np.unique(dat.nrep) #nrep
id_subj_labels = np.unique(dat.id_subj) #npar


axes[1].axvline(0, ls ='--', color= 'black', alpha=0.2)
axes[1].axhline(0.5, ls ='--', color= 'black', alpha=0.2)

for i in id_subj_labels: #for loop to compute the average by each participant
    dat2=dat.loc[(dat.id_subj == i) ,:]
    for cell in nrep_labels:
        sns.set_palette(mycol)
        sns.regplot(x="rDV", y="Responses",  data=dat2.loc[dat2.nrep == cell,:],
           logistic=True, y_jitter=0, scatter_kws={'alpha':0}, ci=True, n_boot=1, ax = axes[1], label=cell,  truncate=True, 
                         line_kws ={'alpha':0.35, 'lw':0.4}); #mean all subject

for cell in nrep_labels: #for loop to compute the plot by the average sample
    sns.set_palette(mycol)
    sns.regplot(x="rDV", y="Responses",  data=dat.loc[(dat.nrep == cell) ,:],
           logistic=True, y_jitter=0, scatter_kws={'alpha':0}, ci=True, n_boot=1,   ax = axes[1], label=cell,  truncate=True, 
                     line_kws ={'lw':2.4});
#sns.despine(offset=1, trim=True, ax = axes[j]);


axes[1].set_ylabel('P(diagonal)', fontsize = 10, labelpad=10); #axes[j].set_yticks(np.arange(0, 1.1, step=0.25), fontsize = 15) 
axes[1].set_xlabel('Decision Variable', fontdict={'size':10}); 

#xticks = [-0.6,-0.3,0,0.3,0.6] #np.round(np.arange(-0.9, 0.6, step=0.3),decimals = 2)
#yticks = np.arange(-0., 1.1, step=0.25)

axes[1].set_xlim(-0.72,0.62)
axes[1].set_xticks([-0.6,0,0.6]) 
axes[1].tick_params(axis='x', labelsize=10)  

axes[1].set_yticks([0.0,0.5,1]) 
axes[1].set_ylim(-0.1,1.05)
axes[1].tick_params(axis='y', labelsize=10) 

sns.despine(ax= axes[1], offset=0.5, trim = True);
#'lower right', borderpad=0.1,prop={'size':10}

lines = [Line2D([0], [0], color=c, linewidth=4) for c in mycol]; labels = ['P1', 'P2', 'P3']; 
axes[1].legend(lines, labels,loc = 'lower right', fontsize=8, framealpha= 0.0)
fig.tight_layout()
#plt.savefig('deci_x_nrep.png',bbox_inches='tight')

# plotting dprime and criterion 



dat = group_sdt_avg[group_sdt_avg.exp_ID == EXP]
g = sns.barplot(data=dat, x="nrep", y="dprime", hue="nrep", ax = axes[2],alpha = 0.9, dodge=dodge,linewidth = 0.6)
g = sns.stripplot(data=dat, x="nrep", y="dprime", hue="nrep", ax = axes[2],alpha = 0.5, dodge=dodge,edgecolor = "white", linewidth=0.6, jitter = 0.2, legend = False)
g.set_xlabel("nrep",fontsize=10)
g.set_ylabel("dprime",fontsize=10)
g.legend(loc='upper left', fontsize=0,  framealpha= 0.0) #labels=['P1', 'P2', 'P3'], #
if g.get_legend():  # Check if a legend exists
    g.get_legend().remove()


axes[2].set_ylim(-0.1,1.85)
axes[2].set_yticks([0.0,0.5,1, 1.5]) 
axes[2].tick_params(axis='y', labelsize=10) 

axes[2].set_xticks([0,1,2])
axes[2].set_xticklabels(['P1','P2','P3'])
axes[2].tick_params(axis='x', labelsize=10) 
axes[2].set_xlabel('Repetition', fontdict={'size':10}); 

sns.despine(offset = .5,  trim=True, ax = axes[2]);

#g.set_axis_labels( axislabels['xlabel'] , axislabels['ylabel'], fontsize = 15 )

dat = group_sdt_avg[group_sdt_avg.exp_ID == EXP]

g = sns.barplot(data=dat, x="nrep", y="c", hue="nrep", ax = axes[3],alpha = 0.9, dodge=dodge,linewidth = 0.6)
g = sns.stripplot(data=dat, x="nrep", y="c", hue="nrep", ax = axes[3],alpha = 0.5, dodge=dodge,edgecolor = "white", linewidth=0.6,jitter = 0.2, legend = False)
# y axis format
axes[3].set_ylim(-1.1, 0.1)
axes[3].set_yticks([-1,-0.5,0]) 
axes[3].tick_params(axis='y', labelsize=10) 

axes[3].set_xticks([0,1,2])
axes[3].set_xticklabels(['P1','P2','P3'])
axes[3].tick_params(axis='x', labelsize=10) 
axes[3].set_xlabel('Repetition', fontdict={'size':10}); 

# x axis format
#axes[2].tick_params(axis='x', labelsize=10) 
sns.despine(offset = .5,  trim=True, ax = axes[3]);
g.set_xlabel("nrep",fontsize=10)
g.set_ylabel("c",fontsize=10)
g.legend(loc='upper left', fontsize=0,  framealpha= 0.0) #labels=['P1', 'P2', 'P3'], 
#g.set(yaxis['ylim'], yaxis['yticks'])   



## Previous responses effect

In [ ]:

sns.set(font_scale = 1, style = 'ticks') 
mycol = colpal3
sns.set_palette(mycol)

titles = ['P1', 'P2', 'P3']
pre_deci_labels = np.unique(df['response1'])     #pre_deci
nreps = np.unique(df['nrep'])     #nreps


dat = df[df.exp_ID == EXP]

nrep_labels = np.unique(dat.nrep) #nrep
id_subj_labels = np.unique(dat.id_subj) #npar


#df_subj = df[df.subj != 's22'] # s16 does not converge

fig, axes = plt.subplots(1, 4, figsize=(12, 3), gridspec_kw={'width_ratios': [.20, .20,.20,.30]})#fig.suptitle("DV for previous decision for the different repetitions", fontsize=20)

labels = ['(n-1) Cardinal', '(n-1) Diagonal']; 
lines = [Line2D([0], [0], color=c, linewidth=2) for c in mycol]; 
axes[0].legend(lines, labels,loc = 'lower right', fontsize=8, framealpha= 0.0)

for j in nreps:
    axes[j].axvline(0, ls='--', color= 'black', alpha = 0.2)
    axes[j].axhline(0.5, ls='--', color= 'black', alpha = 0.2)
    for cell in pre_deci_labels:
        for i in id_subj_labels: #this for loop makes the plot for each participant 
            dati = dat[(dat.id_subj == i) & (dat['response1'] == cell) & (dat.nrep == j)].copy()#df.loc[(df.npar == i) & (df['deci-1'] == cell) & (df.nrep == j),:]
            sns.regplot(x="rDV", y="Responses",  data= dati,
               logistic=True, y_jitter=0, scatter_kws={'alpha':0}, ci=True, n_boot=1,  
                              label=cell,  truncate=True, line_kws ={'alpha':0.35, 'lw':0.7}, ax = axes[j], color = mycol[cell]);

        datall = dat[(dat['response1'] == cell) & (dat.nrep == j) ].copy()
        sns.regplot(x="rDV", y="Responses",  data= datall,
               logistic=True, y_jitter=.0, scatter_kws={'alpha':0}, ci=True, n_boot=1,  
                          label=cell, truncate=True, line_kws={'lw':2.4}, ax = axes[j], color = mycol[cell]);

    # Tweaking subplots
    axes[j].set_title(titles[j],fontsize = 10)

    axes[j].set_xlabel('DV', fontsize = 10)
    axes[j].set_ylabel('P(Diagonal)', fontsize = 10)

    axes[j].set_xlim(-0.72,0.62)
    axes[j].set_xticks([-0.6,0,0.6]) 

    axes[j].tick_params(axis='x', labelsize=10)   
    axes[j].set_yticks([0.0,0.5,1]) 
    axes[j].set_ylim(-0.1,1.05)
    axes[j].tick_params(axis='y', labelsize=10) 
    #axes[j].xticks(np.arange(-0.6, 0.61, step=0.3), fontsize = 25); 
    sns.despine(ax= axes[j], offset=0.5, trim = True);


fig.tight_layout()
##axes[0].ylabel('p(diagonal)', fontsize = 20, labelpad=20); axes[0].yticks(np.arange(0, 1.1, step=0.25), fontsize = 25); plt.xlabel(' ', fontsize = 0)


sizepoint = 4
rt = "v"; pal = pal; sigma = .5
dodge = False
pal = colpal2

dat = group_repeat_correct[group_repeat_correct.exp_ID == EXP]
colpal4 = ['crimson', 'mediumaquamarine']
g = sns.barplot(data=dat, x="nrep", y="RepResponses", hue="accuracy", palette = colpal4, ax = axes[3],alpha = 0.9, dodge=True,linewidth = 0.6)
g = sns.stripplot(data=dat, x="nrep", y="RepResponses", hue="accuracy", palette = colpal4, ax = axes[3],alpha = 0.5, dodge=True,edgecolor = "white", linewidth=0.6,jitter = 0.2, legend = False)
# y axis format
axes[3].set_ylim(-0.1, 1.1)
axes[3].set_yticks([0,0.5,1]) 
axes[3].tick_params(axis='y', labelsize=10) 

axes[3].set_xticks([0,1,2])
axes[3].set_xticklabels(['P1','P2','P3'])
axes[3].tick_params(axis='x', labelsize=10) 
axes[3].set_xlabel('Repetition', fontdict={'size':10}); 

# x axis format
#axes[2].tick_params(axis='x', labelsize=10) 
sns.despine(offset = .5,  trim=True, ax = axes[3]);
g.set_xlabel("nrep",fontsize=10)
g.set_ylabel("P(Repeat)",fontsize=10)
#g.legend(loc='upper left', labels = ['correct', 'incorrect'], fontsize=5,  framealpha= 0.0) #labels=['P1', 'P2', 'P3'], 
#g.set(yaxis['ylim'], yaxis['yticks'])   


# modify the legend
handles, labels = g.get_legend_handles_labels()
new_labels = ['Incorrect', 'Correct']
g.legend(handles, new_labels, loc='upper left', title='', fontsize=6, frameon=True, framealpha=0, facecolor='white')


## Experiment CJ

In [ ]:

dat = group_sdt_avg[group_sdt_avg.exp_ID == EXP]

In [ ]:
dat = group_sdt_avg[group_sdt_avg.exp_ID == EXP]
dat['nrep'] = dat['nrep'].cat.remove_unused_categories()

In [ ]:
dat['nrep'].unique()

In [ ]:
EXP = 'CJ'
# plotting accurary 


sizepoint = 4
rt = "v"; sigma = .5
dodge = False

mycol = colpal2


fig, axes = plt.subplots(1, 4, figsize=(12, 2.7))
fig.subplots_adjust(wspace=0.25, hspace=0)

sns.set_palette(colpal2)

dat = group_avg_correct[group_avg_correct.exp_ID == EXP]
dat = dat[dat['nrep']!=2]
dat['nrep'] = dat['nrep'].cat.remove_unused_categories()


g = sns.barplot(data=dat, x="nrep", y="accuracy", hue="nrep", ax = axes[0],alpha = 0.9, dodge=dodge,linewidth = 0.6)
g = sns.stripplot(data=dat, x="nrep", y="accuracy", hue="nrep", ax = axes[0],alpha = 0.5, dodge=dodge,edgecolor = "white", linewidth=0.6, jitter = 0.2, legend = False)
g.set_xlabel("nrep",fontsize=10)
g.set_ylabel("P(correct)",fontsize=10)
g.legend(loc='upper left', fontsize=0,  framealpha= 0.0) #labels=['P1', 'P2', 'P3'], #


axes[0].set_ylim(-0.1,1.1)
axes[0].set_yticks([0.0,0.5,1]) 
axes[0].tick_params(axis='y', labelsize=10) 
axes[0].set_xlim(-0.75,1.75)

axes[0].set_xticks([0,1])
axes[0].set_xticklabels(['P1','P2'])
axes[0].tick_params(axis='x', labelsize=10) 
axes[0].set_xlabel('Repetition', fontdict={'size':10}); 

sns.despine(offset = .5,  trim=True, ax = axes[0]);



dat = df[df.exp_ID == EXP]
nrep_labels = np.unique(dat.nrep) #nrep
id_subj_labels = np.unique(dat.id_subj) #npar


axes[1].axvline(0, ls ='--', color= 'black', alpha=0.2)
axes[1].axhline(0.5, ls ='--', color= 'black', alpha=0.2)

for i in id_subj_labels: #for loop to compute the average by each participant
    dat2=dat.loc[(dat.id_subj == i) ,:]
    for cell in nrep_labels:
        sns.set_palette(colpal2)
        sns.regplot(x="rDV", y="Responses",  data=dat2.loc[dat2.nrep == cell,:],
           logistic=True, y_jitter=0,  color = colpal2[cell], scatter_kws={'alpha':0}, ci=True, n_boot=1, ax = axes[1], label=cell,  truncate=True, 
                         line_kws ={'alpha':0.35, 'lw':0.4}); #mean all subject

for cell in nrep_labels: #for loop to compute the plot by the average sample
    sns.set_palette(colpal2)
    sns.regplot(x="rDV", y="Responses",  data=dat.loc[(dat.nrep == cell) ,:],
           logistic=True, y_jitter=0,  color = colpal2[cell], scatter_kws={'alpha':0}, ci=True, n_boot=1,   ax = axes[1], label=cell,  truncate=True, 
                     line_kws ={'lw':2.4});
#sns.despine(offset=1, trim=True, ax = axes[j]);


axes[1].set_ylabel('P(diagonal)', fontsize = 10, labelpad=10); #axes[j].set_yticks(np.arange(0, 1.1, step=0.25), fontsize = 15) 
axes[1].set_xlabel('Decision Variable', fontdict={'size':10}); 

#xticks = [-0.6,-0.3,0,0.3,0.6] #np.round(np.arange(-0.9, 0.6, step=0.3),decimals = 2)
#yticks = np.arange(-0., 1.1, step=0.25)

axes[1].set_xlim(-0.72,0.72)
axes[1].set_xticks([-0.6,0,0.6]) 
axes[1].tick_params(axis='x', labelsize=10)  

axes[1].set_yticks([0.0,0.5,1]) 
axes[1].set_ylim(-0.1,1.05)
axes[1].tick_params(axis='y', labelsize=10) 

sns.despine(ax= axes[1], offset=0.5, trim = True);
#'lower right', borderpad=0.1,prop={'size':10}

lines = [Line2D([0], [0], color=c, linewidth=4) for c in colpal2]; labels = ['P1', 'P2']; 
axes[1].legend(lines, labels,loc = 'lower right', fontsize=8, framealpha= 0.0)

#plt.savefig('deci_x_nrep.png',bbox_inches='tight')

# plotting dprime and criterion 

sns.set_palette(colpal2)

dat = group_sdt_avg[group_sdt_avg.exp_ID == EXP]
dat['nrep'] = dat['nrep'].cat.remove_unused_categories()

g = sns.barplot(data=dat, x="nrep", y="dprime", hue="nrep", ax = axes[2],alpha = 0.9, dodge=dodge,linewidth = 0.6)
g = sns.stripplot(data=dat, x="nrep", y="dprime", hue="nrep", ax = axes[2],alpha = 0.5, dodge=dodge,edgecolor = "white", linewidth=0.6, jitter = 0.2, legend = False)
g.set_xlabel("nrep",fontsize=10)
g.set_ylabel("dprime",fontsize=10)
g.legend(loc='upper left', fontsize=0,  framealpha= 0.0) #labels=['P1', 'P2', 'P3'], #


axes[2].set_ylim(-0.1,1.85)
axes[2].set_yticks([0.0,0.5,1, 1.5]) 
axes[2].tick_params(axis='y', labelsize=10) 

axes[2].set_xlim(-0.75,1.75)
axes[2].set_xticks([0,1])
axes[2].set_xticklabels(['P1','P2'])
axes[2].tick_params(axis='x', labelsize=10) 
axes[2].set_xlabel('Repetition', fontdict={'size':10}); 

sns.despine(offset = .5,  trim=True, ax = axes[2]);

#g.set_axis_labels( axislabels['xlabel'] , axislabels['ylabel'], fontsize = 15 )

dat = group_sdt_avg[group_sdt_avg.exp_ID == EXP]
dat['nrep'] = dat['nrep'].cat.remove_unused_categories()

g = sns.barplot(data=dat, x="nrep", y="c", hue="nrep", ax = axes[3],alpha = 0.9, dodge=dodge,linewidth = 0.6)
g = sns.stripplot(data=dat, x="nrep", y="c", hue="nrep", ax = axes[3],alpha = 0.5, dodge=dodge,edgecolor = "white", linewidth=0.6,jitter = 0.2, legend = False)
# y axis format
axes[3].set_ylim(-1.1, 0.1)
axes[3].set_yticks([-1,-0.5,0]) 
axes[3].tick_params(axis='y', labelsize=10) 

axes[3].set_xlim(-0.75,1.75)
axes[3].set_xticks([0,1])
axes[3].set_xticklabels(['P1','P2'])
axes[3].tick_params(axis='x', labelsize=10) 
axes[3].set_xlabel('Repetition', fontdict={'size':10}); 

# x axis format
#axes[2].tick_params(axis='x', labelsize=10) 
sns.despine(offset = .5,  trim=True, ax = axes[3]);
g.set_xlabel("nrep",fontsize=10)
g.set_ylabel("c",fontsize=10)
g.legend(loc='upper left', fontsize=0,  framealpha= 0.0) #labels=['P1', 'P2', 'P3'], 
#g.set(yaxis['ylim'], yaxis['yticks'])   
fig.tight_layout(pad=0.5)
#figpath = os.path.join(figures_path, 'CJ_performance.pdf')
plt.savefig(figpath ,bbox_inches='tight', dpi = 300)


## Comparing accuracy repeat non repeat

In [ ]:
groupCJ_avg_correct = df[df.exp_ID == 'CJ'].groupby(['trial_type', 'id_subj','nrep'],as_index=False, observed = True)[['accuracy']].mean()
groupCJ_sdt_avg = df[df.exp_ID == 'CJ'].groupby(['trial_type', 'id_subj','nrep'], observed = True).apply(sdt)
groupCJ_sdt_avg.reset_index(inplace=True)
groupCJ_repeat_correct = df[df.exp_ID == 'CJ'].groupby(['trial_type', 'id_subj','nrep', 'accuracy'], observed = True,as_index=False)[['RepResponses']].mean()

In [ ]:
groupCJ_avg_correct_long = groupCJ_avg_correct[groupCJ_avg_correct.trial_type == 'repeat']
groupCJ_avg_correct_long.reset_index(inplace = True)
aux = groupCJ_avg_correct[groupCJ_avg_correct.trial_type == 'nonrepeat'].accuracy
groupCJ_avg_correct_long['accuracy_nonrep'] = aux

In [ ]:
groupCJ_sdt_avg_long = groupCJ_sdt_avg[groupCJ_sdt_avg.trial_type == 'repeat']
groupCJ_sdt_avg_long.reset_index(inplace = True)
aux = groupCJ_sdt_avg[groupCJ_sdt_avg.trial_type == 'nonrepeat']['dprime']
groupCJ_sdt_avg_long['dprime_nonrep'] = aux

In [ ]:
sizeplot = {}; sizeplot['height'] = 4; sizeplot['aspect'] = 1
axislabels = {}; axislabels['xlabel'] = 'Accuracy (repeated seq.)'; axislabels['ylabel']= 'Accuracy (non repeated seq)'
yaxis = {}; yaxis['ylim']= [0.4,1.1]; yaxis['yticks']=[ 0.5, 0.75, 1]
xaxis = {}; xaxis['xlim']=[0.4,1.1]; xaxis['xticks']=[0.5,  0.75,  1]

dx = 'accuracy'; dy = 'accuracy_nonrep'; hue = 'nrep';  row = None; col = None
pal = colpal2

dat = groupCJ_avg_correct_long.copy()

simplescatterplot(dat, dx, dy, hue, col, row, pal, sizeplot, yaxis, xaxis, axislabels, sizepoint = 5,  refline = 0.5)

In [ ]:
# function that I used to plot multiple data
def simplescatterplot(data, dx, dy, hue, col, row, pal, size, yaxis, xaxis, axislabels, sizepoint, refline):    
    def add_ref_lines(x, y, **kwargs):
        plt.axvline(x=x,  **kwargs)
        plt.axhline(y=y,  **kwargs)
        plt.plot([-1,3],[-1,3], 'black', linewidth=1, alpha = 0.5)
         
    ort = "v"; pal = pal; sigma = .5
    g = sns.FacetGrid(data ,  row = row, col = col, height= size['height'], aspect=size['aspect'], margin_titles=True) # col="nrep",    
    g.map(add_ref_lines, x = refline, y = refline, color = 'black', alpha=0.2,ls='--')

    if sizepoint == None:
        sizepoint = 3
    
    g.set(ylim = yaxis['ylim'], yticks = yaxis['yticks'])
    g.set(xlim = xaxis['xlim'], xticks = xaxis['xticks'])
        
    g.map_dataframe(sns.scatterplot, x=dx, y=dy, hue=hue, palette = pal, size = sizepoint, sizes = (60,100), alpha = 0.5 )

    g.add_legend()
    axes = g.axes
    # modify the legend


    #g.legend( new_labels, loc='upper left', title='', fontsize=6, frameon=True, framealpha=0, facecolor='white')
   
    g.set_axis_labels( axislabels['xlabel'] , axislabels['ylabel'], fontsize = 12 )
    #g.tight_layout()


In [ ]:
EXP = 'CJ'
# plotting accurary 


sizepoint = 4
rt = "v"; sigma = .5
dodge = False

mycol = colpal2
pal =  colpal2
#dat = df[df.exp_ID == EXP]


fig, axes = plt.subplots(2, 4, figsize=(12, 6), gridspec_kw={'width_ratios': [.28, 0.25, .22,.22]})

repeat = ['nonrepeat','repeat']
for irep, irepeat in enumerate(repeat):
    sns.set_palette(colpal2)
    dat = groupCJ_avg_correct[(groupCJ_avg_correct.trial_type == irepeat) ]

    g = sns.barplot(data=dat, x="nrep", y="accuracy", hue="nrep", ax = axes[irep,0],alpha = 0.9, dodge=dodge,linewidth = 0.6)
    g = sns.stripplot(data=dat, x="nrep", y="accuracy", hue="nrep", ax = axes[irep,0],alpha = 0.5, dodge=dodge,edgecolor = "white", linewidth=0.6, jitter = 0.2, legend = False)
    g.set_xlabel("nrep",fontsize=10)
    g.set_ylabel("P(correct)",fontsize=10)
    g.legend(loc='upper left', fontsize=0,  framealpha= 0.0) #labels=['P1', 'P2', 'P3'], #


    axes[irep, 0].set_ylim(-0.1,1.1)
    axes[irep, 0].set_yticks([0.0,0.5,1]) 
    axes[irep, 0].tick_params(axis='y', labelsize=10) 

     # lateral title
    axes[irep, 0].set_title(irepeat,  rotation='vertical',x=-0.4,y=0.25, fontdict={'size':15})
        

    axes[irep, 0].set_xticks([0,1])
    axes[irep, 0].set_xticklabels(['P1','P2'])
    axes[irep, 0].tick_params(axis='x', labelsize=10) 
    axes[irep, 0].set_xlabel('Repetition', fontdict={'size':10}); 

    sns.despine(offset = .5,  trim=True, ax = axes[irep, 0]);



    dat = df[(df.exp_ID == EXP) & (df.trial_type == irepeat)]
    nrep_labels = np.unique(dat.nrep) #nrep
    id_subj_labels = np.unique(dat.id_subj) #npar


    axes[irep, 1].axvline(0, ls ='--', color= 'black', alpha=0.2)
    axes[irep, 1].axhline(0.5, ls ='--', color= 'black', alpha=0.2)

    for i in id_subj_labels: #for loop to compute the average by each participant
        dat2=dat.loc[(dat.id_subj == i) ,:]
        for cell in nrep_labels:
            sns.set_palette(colpal2)
            sns.regplot(x="rDV", y="Responses",  data=dat2.loc[dat2.nrep == cell,:],
            logistic=True, y_jitter=0, scatter_kws={'alpha':0}, ci=True, n_boot=1, ax = axes[irep, 1], label=cell,  truncate=True, 
                            line_kws ={'alpha':0.35, 'lw':0.4}, color = colpal2[cell]); #mean all subject

    for cell in nrep_labels: #for loop to compute the plot by the average sample
        sns.set_palette(pal)
        sns.regplot(x="rDV", y="Responses",  data=dat.loc[(dat.nrep == cell) ,:],
            logistic=True, y_jitter=0, color = colpal2[cell], scatter_kws={'alpha':0}, ci=True, n_boot=1,   ax = axes[irep, 1], label=cell,  truncate=True, 
                        line_kws ={'lw':2.4});
    #sns.despine(offset=1, trim=True, ax = axes[j]);


    axes[irep, 1].set_ylabel('P(diagonal)', fontsize = 10, labelpad=10); #axes[j].set_yticks(np.arange(0, 1.1, step=0.25), fontsize = 15) 
    axes[irep, 1].set_xlabel('Decision Variable', fontdict={'size':10}); 

    #xticks = [-0.6,-0.3,0,0.3,0.6] #np.round(np.arange(-0.9, 0.6, step=0.3),decimals = 2)
    #yticks = np.arange(-0., 1.1, step=0.25)

    axes[irep, 1].set_xlim(-0.72,0.62)
    axes[irep, 1].set_xticks([-0.6,0,0.6]) 
    axes[irep, 1].tick_params(axis='x', labelsize=10)  

    axes[irep, 1].set_yticks([0.0,0.5,1]) 
    axes[irep, 1].set_ylim(-0.1,1.05)
    axes[irep, 1].tick_params(axis='y', labelsize=10) 

    sns.despine(ax= axes[irep, 1], offset=0.5, trim = True);
    #'lower right', borderpad=0.1,prop={'size':10}

    lines = [Line2D([0], [0], color=c, linewidth=4) for c in mycol]; labels = ['P1', 'P2']; 
    axes[irep, 1].legend(lines, labels,loc = 'lower right', fontsize=8, framealpha= 0.0)
    fig.tight_layout()
    #plt.savefig('deci_x_nrep.png',bbox_inches='tight')

    # plotting dprime and criterion 

    sns.set_palette(colpal2)

    dat = groupCJ_sdt_avg[(groupCJ_sdt_avg.trial_type == irepeat) ]
    g = sns.barplot(data=dat, x="nrep", y="dprime", hue="nrep", ax = axes[irep, 2],alpha = 0.9, dodge=dodge,linewidth = 0.6)
    g = sns.stripplot(data=dat, x="nrep", y="dprime", hue="nrep", ax = axes[irep, 2],alpha = 0.5, dodge=dodge,edgecolor = "white", linewidth=0.6, jitter = 0.2, legend = False)
    g.set_xlabel("nrep",fontsize=10)
    g.set_ylabel("dprime",fontsize=10)
    g.legend(loc='upper left', fontsize=0,  framealpha= 0.0) #labels=['P1', 'P2', 'P3'], #


    axes[irep, 2].set_ylim(-0.1,1.85)
    axes[irep, 2].set_yticks([0.0,0.5,1, 1.5]) 
    axes[irep, 2].tick_params(axis='y', labelsize=10) 

    axes[irep, 2].set_xticks([0,1])
    axes[irep, 2].set_xticklabels(['P1','P2'])
    axes[irep, 2].tick_params(axis='x', labelsize=10) 
    axes[irep, 2].set_xlabel('Repetition', fontdict={'size':10}); 

    sns.despine(offset = .5,  trim=True, ax = axes[irep, 2]);

    #g.set_axis_labels( axislabels['xlabel'] , axislabels['ylabel'], fontsize = 15 )

  

    g = sns.barplot(data=dat, x="nrep", y="c", hue="nrep", ax = axes[irep, 3],alpha = 0.9, dodge=dodge,linewidth = 0.6)
    g = sns.stripplot(data=dat, x="nrep", y="c", hue="nrep", ax = axes[irep, 3],alpha = 0.5, dodge=dodge,edgecolor = "white", linewidth=0.6,jitter = 0.2, legend = False)
    # y axis format
    axes[irep, 3].set_ylim(-1.1, 0.1)
    axes[irep, 3].set_yticks([-1,-0.5,0]) 
    axes[irep, 3].tick_params(axis='y', labelsize=10) 

    axes[irep, 3].set_xticks([0,1])
    axes[irep, 3].set_xticklabels(['P1','P2'])
    axes[irep, 3].tick_params(axis='x', labelsize=10) 
    axes[irep, 3].set_xlabel('Repetition', fontdict={'size':10}); 

    # x axis format
    #axes[2].tick_params(axis='x', labelsize=10) 
    sns.despine(offset = .5,  trim=True, ax = axes[irep, 3]);
    g.set_xlabel("nrep",fontsize=10)
    g.set_ylabel("c",fontsize=10)
    g.legend(loc='upper left', fontsize=0,  framealpha= 0.0) #labels=['P1', 'P2', 'P3'], 
    #g.set(yaxis['ylim'], yaxis['yticks'])   


## Previous responses effect

In [ ]:

sns.set(font_scale = 1, style = 'ticks') 
mycol = colpal3
sns.set_palette(mycol)


dat = df[df.exp_ID == EXP]

titles = ['P1', 'P2']
pre_deci_labels = np.unique(dat['response1'])     #pre_deci
nreps = np.unique(dat['nrep'])     #nreps
id_subj_labels = np.unique(dat.id_subj) #npar


#df_subj = df[df.subj != 's22'] # s16 does not converge

fig, axes = plt.subplots(1, 3, figsize=(10, 3), gridspec_kw={'width_ratios': [.20, .20,.30]})
#fig, axes = plt.subplots(1, 3, figsize=(9, 3))
#fig.suptitle("DV for previous decision for the different repetitions", fontsize=20)

labels = ['(n-1) Cardinal', '(n-1) Diagonal']; 
lines = [Line2D([0], [0], color=c, linewidth=2) for c in mycol]; 
axes[0].legend(lines, labels,loc = 'lower right', fontsize=8, framealpha= 0.0)

for j in nreps:
    axes[j].axvline(0, ls='--', color= 'black', alpha = 0.2)
    axes[j].axhline(0.5, ls='--', color= 'black', alpha = 0.2)
    for cell in pre_deci_labels:
        for i in id_subj_labels: #this for loop makes the plot for each participant 
            dati = dat[(dat.id_subj == i) & (dat['response1'] == cell) & (dat.nrep == j)].copy()#df.loc[(df.npar == i) & (df['deci-1'] == cell) & (df.nrep == j),:]
            sns.regplot(x="rDV", y="Responses",  data= dati,
               logistic=True, y_jitter=0, scatter_kws={'alpha':0}, ci=True, n_boot=1,  
                              label=cell,  truncate=True, line_kws ={'alpha':0.35, 'lw':0.7}, ax = axes[j], color = mycol[cell]);

        datall = dat[(dat['response1'] == cell) & (dat.nrep == j) ].copy()
        sns.regplot(x="rDV", y="Responses",  data= datall,
               logistic=True, y_jitter=.0, scatter_kws={'alpha':0}, ci=True, n_boot=1,  
                          label=cell, truncate=True, line_kws={'lw':2.4}, ax = axes[j], color = mycol[cell]);

    # Tweaking subplots
    axes[j].set_title(titles[j],fontsize = 10)

    axes[j].set_xlabel('DV', fontsize = 10)
    axes[j].set_ylabel('P(Diagonal)', fontsize = 10)

    axes[j].set_xlim(-0.72,0.62)
    axes[j].set_xticks([-0.6,0,0.6]) 

    axes[j].tick_params(axis='x', labelsize=10)   
    axes[j].set_yticks([0.0,0.5,1]) 
    axes[j].set_ylim(-0.1,1.05)
    axes[j].tick_params(axis='y', labelsize=10) 
    #axes[j].xticks(np.arange(-0.6, 0.61, step=0.3), fontsize = 25); 
    sns.despine(ax= axes[j], offset=0.5, trim = True);


fig.tight_layout()
##axes[0].ylabel('p(diagonal)', fontsize = 20, labelpad=20); axes[0].yticks(np.arange(0, 1.1, step=0.25), fontsize = 25); plt.xlabel(' ', fontsize = 0)


sizepoint = 4
rt = "v"; pal = pal; sigma = .5
dodge = False
pal = colpal2

dat = group_repeat_correct[group_repeat_correct.exp_ID == EXP]
colpal4 = ['crimson', 'mediumaquamarine']
g = sns.barplot(data=dat, x="nrep", y="RepResponses", hue="accuracy", palette = colpal4, ax = axes[2],alpha = 0.9, dodge=True,linewidth = 0.6)
g = sns.stripplot(data=dat, x="nrep", y="RepResponses", hue="accuracy", palette = colpal4, ax = axes[2],alpha = 0.5, dodge=True,edgecolor = "white", linewidth=0.6,jitter = 0.2, legend = False)
# y axis format
axes[2].set_ylim(-0.1, 1.1)
axes[2].set_yticks([0,0.5,1]) 
axes[2].tick_params(axis='y', labelsize=10) 

axes[2].set_xticks([0,1])
axes[2].set_xticklabels(['P1','P2'])
axes[2].tick_params(axis='x', labelsize=10) 
axes[2].set_xlabel('Repetition', fontdict={'size':10}); 

# x axis format
#axes[2].tick_params(axis='x', labelsize=10) 
sns.despine(offset = .5,  trim=True, ax = axes[2]);
g.set_xlabel("nrep",fontsize=10)
g.set_ylabel("P(Repeat)",fontsize=10)
#g.legend(loc='upper left', labels = ['correct', 'incorrect'], fontsize=5,  framealpha= 0.0) #labels=['P1', 'P2', 'P3'], 
#g.set(yaxis['ylim'], yaxis['yticks'])   


# modify the legend
handles, labels = g.get_legend_handles_labels()
new_labels = ['Incorrect', 'Correct']
g.legend(handles, new_labels, loc='upper left', title='', fontsize=6, frameon=True, framealpha=0, facecolor='white')


figpath = os.path.join(figures_path, 'CJ_prev_resp.pdf')
plt.savefig(figpath ,bbox_inches='tight', dpi = 300)
